# CA1 deployment: People Counting Assistant + Twist 2 (cyclists)

The Lesson 4 page, extended for the twist. The reply now counts **pedestrians and
cyclists** moving left and right, and the picture links each rider to their bike.

The service code lives in the team repo (`app/service.py`, `src/counting.py`), so
this notebook only sets up, loads the model once, and launches. Run top to bottom.

**Before pushing this notebook:** *Edit → Clear all outputs*. Outputs can hold photos.

## 1. Setup

The repo is private, so cloning needs a GitHub token. Add one in Colab's
**Secrets** panel (key icon) as `GITHUB_TOKEN`, with notebook access on. A
fine-grained token with *read-only Contents* on `c389-ca1` is enough.
If there is no secret, the cell asks for it.

In [ ]:
ULTRALYTICS_VERSION = "8.4.120"
REPO = "Morn-Thuta/c389-ca1"
BRANCH = "main"   # change to test a PR branch, e.g. "feat/counting-app"

import os
import subprocess
from getpass import getpass

subprocess.run(["pip", "install", "-q", "ultralytics==" + ULTRALYTICS_VERSION, "gradio",
                "imagehash"], check=True)

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
if not token:
    token = getpass("GitHub token (read access to %s): " % REPO)

if not os.path.isdir("/content/c389-ca1"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                    "https://x-access-token:%s@github.com/%s.git" % (token, REPO),
                    "/content/c389-ca1"], check=True)
else:
    subprocess.run(["git", "-C", "/content/c389-ca1", "pull", "-q"], check=True)
del token

os.chdir("/content/c389-ca1")
print("repo at", os.getcwd())

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

os.environ["PROJECT_ROOT"] = "/content/drive/MyDrive/C389_2"

# The baseline from L3. Once twist_3cls is trained, point this at its best.pt:
# the stock bicycle model is then skipped automatically.
WEIGHTS = os.path.join(os.environ["PROJECT_ROOT"], "artifacts", "L3", "best.pt")
BIKE_WEIGHTS = "yolo26n.pt"   # stock COCO model, used for bicycles only
print("weights:", WEIGHTS, "| exists:", os.path.exists(WEIGHTS))

## 2. Load the model once

At startup, never inside the request handler.

In [ ]:
import logging
import sys

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s",
                    stream=sys.stdout, force=True)

from app import service
from app.app import build_demo
from src.config import CONF_THRESHOLD

DETECTOR = service.load_model(WEIGHTS, BIKE_WEIGHTS)
print("mode   :", DETECTOR.mode)
print("classes:", DETECTOR.class_names)

## 3. Launch with a public link

`share=True` gives a `gradio.live` link that anyone holding it can use. It lasts
up to a week and dies when this runtime stops. For the demo, keep this tab open
and have the backup recording ready.

In [ ]:
demo = build_demo()
demo.launch(share=True)

## 4. Testing the error conditions

Same cases as L4, called on `build_reply` directly. Record status and message as
they come back (report §4.2).

In [ ]:
import tempfile
from PIL import Image

scratch = tempfile.mkdtemp()
text_file = os.path.join(scratch, "not_an_image.txt")
open(text_file, "w").write("this is not a photograph")
empty_file = os.path.join(scratch, "empty.jpg")
open(empty_file, "wb").close()
huge_file = os.path.join(scratch, "huge.jpg")
Image.new("RGB", (9000, 7000), (200, 200, 200)).save(huge_file, quality=30)
nobody_file = os.path.join(scratch, "nobody.jpg")
Image.new("RGB", (640, 480), (214, 210, 202)).save(nobody_file)

CASES = [
    ("nothing sent", None),
    ("a text file", text_file),
    ("an empty file", empty_file),
    ("a very large image", huge_file),
    ("an image with nobody in it", nobody_file),
]
rows = []
for label, payload in CASES:
    reply, _ = service.build_reply(payload)
    rows.append((label, reply["status"], reply.get("error") or reply.get("counts")))

print("\n%-28s %-7s %s" % ("case", "status", "what came back"))
for label, status, summary in rows:
    print("%-28s %-7s %s" % (label, status, summary))

## 5. Warm response time

Mean of 5 calls in this already-warm process.

In [ ]:
import time

sample = Image.new("RGB", (640, 480), (214, 210, 202))
warm = []
for _ in range(5):
    started = time.time()
    service.count_people(sample, CONF_THRESHOLD)
    warm.append(time.time() - started)
print("warm inference, average: %.3f s (two models while the stock bike model is in use)"
      % (sum(warm) / len(warm)))

## 6. Close

Run this when finished, or the runtime keeps serving the link.

In [ ]:
demo.close()
print("The share link is closed.")